# Task 6 — Payments Design & Gateway Setup (Quality Baseline)
**Focus:** Baseline match quality before monetization changes behavior.

This is a **standalone notebook**. It re-loads students.csv / jobs.csv / matches.csv and
records a frozen, timestamped snapshot of current match quality — a guardrail to detect
if payments/pricing changes in Week 3 silently degrade matching. No payment logic is built
here (that's Backend's scope) — this is purely the AI/ML quality reference point.

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from sklearn.metrics import precision_score, recall_score, confusion_matrix
print('Libraries loaded')

Libraries loaded


## Cell 2 — Load data

In [2]:
students = pd.read_csv('../datasets/students.csv')
jobs     = pd.read_csv('../datasets/jobs.csv')
matches  = pd.read_csv('../datasets/matches.csv')
print('Students:', students.shape, '| Jobs:', jobs.shape, '| Matches:', matches.shape)

Students: (20, 7) | Jobs: (9, 7) | Matches: (180, 6)


## Cell 3 — Core helper functions (skills, vectors, thresholds, ranking)

In [3]:
def parse_skills(skill_str):
    result = {}
    if pd.isna(skill_str) or str(skill_str).strip() == '':
        return result
    for part in str(skill_str).split(','):
        part = part.strip()
        if ':' in part:
            skill, score = part.split(':', 1)
            try:
                result[skill.strip()] = int(score.strip())
            except ValueError:
                pass
    return result


def compute_skill_overlap(student_skills_str, job_skills_str):
    student_skills = parse_skills(student_skills_str)
    required_skills = parse_skills(job_skills_str)
    if not required_skills:
        return 0, 0.0
    met = sum(1 for skill, req in required_skills.items() if student_skills.get(skill, 0) >= req)
    return met, round(met / len(required_skills), 3)


def compute_experience_gap(internship_months, min_exp_years):
    return round(internship_months / 6.0 - min_exp_years, 2)


def validate_thresholds(student_id, job_id):
    student = students[students['student_id'] == student_id].iloc[0]
    job     = jobs[jobs['job_id'] == job_id].iloc[0]
    student_skills  = parse_skills(student['skills'])
    required_skills = parse_skills(job['required_skills'])
    results = {}
    for skill, required_level in required_skills.items():
        student_level = student_skills.get(skill, 0)
        results[skill] = {'required_level': required_level, 'student_level': student_level,
                           'passed': bool(student_level >= required_level)}
    overall_passed = all(r['passed'] for r in results.values()) if results else False
    return results, overall_passed


all_skills = set()
for s in students['skills'].dropna():
    all_skills.update(parse_skills(s).keys())
for s in jobs['required_skills'].dropna():
    all_skills.update(parse_skills(s).keys())
MASTER_SKILLS = sorted(all_skills)


def skills_to_vector(skill_str):
    skill_dict = parse_skills(skill_str)
    return np.array([skill_dict.get(skill, 0) for skill in MASTER_SKILLS])


students['skill_vector'] = students['skills'].apply(skills_to_vector)
jobs['threshold_vector']  = jobs['required_skills'].apply(skills_to_vector)


def cosine_similarity_manual(vec1, vec2):
    dot = np.dot(vec1, vec2)
    n1, n2 = np.linalg.norm(vec1), np.linalg.norm(vec2)
    return 0.0 if n1 == 0 or n2 == 0 else dot / (n1 * n2)


def rank_students_for_job(job_id, top_n=None):
    j_vec = jobs[jobs['job_id'] == job_id]['threshold_vector'].iloc[0]
    results = []
    for _, student in students.iterrows():
        sim = cosine_similarity_manual(student['skill_vector'], j_vec)
        results.append({'student_id': student['student_id'], 'match_score': round(sim, 3)})
    ranked = pd.DataFrame(results).sort_values('match_score', ascending=False).reset_index(drop=True)
    ranked.index += 1
    return ranked.head(top_n) if top_n else ranked


print('Helper functions ready')

Helper functions ready


## Cell 4 — Compute Match Decision for Every Pair (current system behavior)

In [4]:
def get_decision(student_id, job_id):
    student = students[students['student_id'] == student_id].iloc[0]
    job = jobs[jobs['job_id'] == job_id].iloc[0]
    _, overlap_ratio = compute_skill_overlap(student['skills'], job['required_skills'])
    exp_gap = compute_experience_gap(student['internship_months'], job['min_experience_years'])
    return 1 if (overlap_ratio >= 0.6 and exp_gap >= -0.5) else 0


matches['predicted_decision'] = matches.apply(
    lambda row: get_decision(row['student_id'], row['job_id']), axis=1
)
print(matches[['student_id', 'job_id', 'label', 'predicted_decision']].head(10))

   student_id  job_id  label  predicted_decision
0           1     101      1                   1
1           1     102      0                   0
2           1     103      0                   0
3           1     104      1                   1
4           1     105      0                   0
5           1     106      0                   0
6           1     107      0                   0
7           1     108      0                   0
8           1     109      0                   0
9           2     101      0                   0


## Cell 5 — Decision-Logic Consistency Check (not an independent quality metric)

**Honesty note:** `predicted_decision` and `matches.csv`'s `label` use the SAME formula
(overlap_ratio >= 0.6 AND exp_gap >= -0.5). So precision/recall here will always be 1.0 —
this just confirms there's no bug in the decision function, not that match quality is
actually good. The real baseline metric is **ranking quality** (Cell 6/7), since ranking
by cosine similarity is a genuinely independent method from the labeling rule.

In [5]:
# SANITY CHECK ONLY — see honesty note above. Will show 1.0 by construction.
y_true = matches['label'].values
y_pred = matches['predicted_decision'].values

consistency_precision = precision_score(y_true, y_pred, zero_division=0)
consistency_recall = recall_score(y_true, y_pred, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

if cm.size == 4:
    tn, fp, fn, tp = cm.ravel()
    consistency_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
else:
    consistency_fpr = 0.0

print('=== DECISION-LOGIC CONSISTENCY (sanity check, not the real baseline) ===')
print(f'Precision: {consistency_precision:.3f} | Recall: {consistency_recall:.3f} | FPR: {consistency_fpr:.3f}')
print('(Expected to be 1.0/1.0/0.0 since decision and label use the same rule — confirms no bug.)')

=== DECISION-LOGIC CONSISTENCY (sanity check, not the real baseline) ===
Precision: 1.000 | Recall: 1.000 | FPR: 0.000
(Expected to be 1.0/1.0/0.0 since decision and label use the same rule — confirms no bug.)


## Cell 6 — Ranking Quality (Precision@K / Recall@K) — per job breakdown

Per-job breakdown, not just one combined number — required by the brief's
'no breakdown by segment' pitfall warning.

In [6]:
def precision_recall_at_k(k=3):
    rows = []
    for job_id in jobs['job_id'].unique():
        true_positive_students = set(
            matches[(matches['job_id'] == job_id) & (matches['label'] == 1)]['student_id']
        )
        if not true_positive_students:
            continue
        ranked = rank_students_for_job(job_id, top_n=k)
        shortlisted = set(ranked['student_id'])
        hits = len(shortlisted & true_positive_students)
        rows.append({
            'job_id': job_id,
            'job_title': jobs[jobs['job_id'] == job_id]['job_title'].iloc[0],
            'precision_at_k': round(hits / k, 3),
            'recall_at_k': round(hits / len(true_positive_students), 3)
        })
    return pd.DataFrame(rows)


per_job_ranking = precision_recall_at_k(k=3)
print('=== PER-JOB RANKING QUALITY (K=3) ===')
print(per_job_ranking.to_string(index=False))
print()
print(f"AVERAGE Precision@3: {per_job_ranking['precision_at_k'].mean():.3f}")
print(f"AVERAGE Recall@3   : {per_job_ranking['recall_at_k'].mean():.3f}")

=== PER-JOB RANKING QUALITY (K=3) ===
 job_id          job_title  precision_at_k  recall_at_k
    101       Data Analyst           1.000        0.375
    102  Backend Developer           1.000        1.000
    103        ML Engineer           0.333        1.000
    104         BI Analyst           1.000        0.750
    105 Frontend Developer           0.667        1.000
    106     Cloud Engineer           0.333        1.000
    107   Security Analyst           0.333        1.000
    108  Android Developer           0.333        1.000
    109  Software Engineer           0.333        1.000

AVERAGE Precision@3: 0.592
AVERAGE Recall@3   : 0.903


## Cell 7 — Ranking Quality by Skill Domain (genuine segment breakdown)

Groups jobs by their primary required skill and reports **Precision@K/Recall@K**
(the independent ranking metric) per domain — not the circular decision-vs-label check.
This shows whether ranking quality is uneven across domains (e.g. Data roles vs Backend roles).

In [7]:
def get_primary_skill(required_skills_str):
    skills = parse_skills(required_skills_str)
    return max(skills, key=skills.get) if skills else 'Unknown'


jobs['primary_skill'] = jobs['required_skills'].apply(get_primary_skill)
ranking_with_domain = per_job_ranking.merge(
    jobs[['job_id', 'primary_skill']], on='job_id', how='left'
)

domain_df = ranking_with_domain.groupby('primary_skill').agg(
    jobs_in_domain=('job_id', 'count'),
    avg_precision_at_3=('precision_at_k', 'mean'),
    avg_recall_at_3=('recall_at_k', 'mean')
).round(3).reset_index()

print('=== RANKING QUALITY BY SKILL DOMAIN (genuine, non-circular) ===')
print(domain_df.to_string(index=False))

=== RANKING QUALITY BY SKILL DOMAIN (genuine, non-circular) ===
primary_skill  jobs_in_domain  avg_precision_at_3  avg_recall_at_3
         .NET               1               0.333            1.000
          AWS               1               0.333            1.000
Cybersecurity               1               0.333            1.000
        Excel               1               1.000            0.750
         Java               1               1.000            1.000
   JavaScript               1               0.667            1.000
       Kotlin               1               0.333            1.000
       Python               2               0.666            0.688


## Cell 8 — Live Demo: One Real Example Walkthrough

In [8]:
demo_student_id = students['student_id'].iloc[0]
demo_job_id = jobs['job_id'].iloc[0]

student = students[students['student_id'] == demo_student_id].iloc[0]
job = jobs[jobs['job_id'] == demo_job_id].iloc[0]
count, ratio = compute_skill_overlap(student['skills'], job['required_skills'])
gap = compute_experience_gap(student['internship_months'], job['min_experience_years'])
decision = get_decision(demo_student_id, demo_job_id)

print('=== LIVE BASELINE DEMO ===')
print(f'Student {demo_student_id} vs Job {demo_job_id} ({job["job_title"]})')
print(f'Skill overlap: {count} skills met, ratio = {ratio}')
print(f'Experience gap: {gap} years')
print(f'Decision: {"MATCH" if decision == 1 else "NO_MATCH"}')
print()
print('Plain-English: this is what match quality looks like RIGHT NOW, before any')
print('pricing or payment-gated features change student/company behavior.')

=== LIVE BASELINE DEMO ===
Student 1 vs Job 101 (Data Analyst)
Skill overlap: 3 skills met, ratio = 1.0
Experience gap: 2.0 years
Decision: MATCH

Plain-English: this is what match quality looks like RIGHT NOW, before any
pricing or payment-gated features change student/company behavior.


## Cell 9 — Edge Case Handling Check

Confirms the baseline metrics don't break or silently misreport when data is sparse —
important since once real money is involved, a silent failure here is costly.

In [9]:
edge_case_log = []

# Edge case: job with zero true positives
for job_id in jobs['job_id'].unique():
    true_pos = matches[(matches['job_id'] == job_id) & (matches['label'] == 1)]
    if len(true_pos) == 0:
        edge_case_log.append(f'Job {job_id} has zero true positive students — excluded from Precision@K (handled, not crashed)')

# Edge case: confirm no NaN/missing decisions
missing_decisions = matches['predicted_decision'].isna().sum()
edge_case_log.append(f'Missing/NaN predicted decisions: {missing_decisions} (should be 0)')

if not edge_case_log:
    edge_case_log.append('No edge cases triggered in this run')

for log in edge_case_log:
    print('-', log)

- Missing/NaN predicted decisions: 0 (should be 0)


## Cell 10 — Save the Frozen Baseline Snapshot (the actual deliverable)

This is what gets handed off for Week-3 monitoring. It is timestamped and versioned
so future runs (post-payments) can be compared against THIS exact reference point.

In [10]:
baseline_snapshot = {
    'baseline_version': 'v1_pre_payments',
    'recorded_at': datetime.now().isoformat(),
    'dataset_size': {
        'students': len(students),
        'jobs': len(jobs),
        'pairs_evaluated': len(matches)
    },
    'decision_logic_consistency_check': {
        'precision': round(consistency_precision, 3),
        'recall': round(consistency_recall, 3),
        'false_positive_rate': round(consistency_fpr, 3),
        'note': 'Sanity check only — expected 1.0 by construction, confirms no bug'
    },
    'ranking_quality_avg': {
        'precision_at_3': round(per_job_ranking['precision_at_k'].mean(), 3),
        'recall_at_3': round(per_job_ranking['recall_at_k'].mean(), 3)
    },
    'per_job_ranking_breakdown': per_job_ranking.to_dict(orient='records'),
    'per_domain_ranking_breakdown': domain_df.to_dict(orient='records'),
    'edge_cases_checked': edge_case_log,
    'purpose': 'Reference snapshot to detect match-quality regressions after Week-3 payment/pricing changes launch.',
    'how_to_use': 'Re-run this notebook after payments go live; compare new Precision@3/Recall@3 (the genuine ranking metric) against this snapshot. A meaningful drop signals payments changed behavior in a way that hurt match quality.'
}

with open('task6_quality_baseline.json', 'w') as f:
    json.dump(baseline_snapshot, f, indent=2)

print('=== BASELINE SNAPSHOT SAVED: task6_quality_baseline.json ===')
print(json.dumps(baseline_snapshot, indent=2))

=== BASELINE SNAPSHOT SAVED: task6_quality_baseline.json ===
{
  "baseline_version": "v1_pre_payments",
  "recorded_at": "2026-06-24T13:05:48.013053",
  "dataset_size": {
    "students": 20,
    "jobs": 9,
    "pairs_evaluated": 180
  },
  "decision_logic_consistency_check": {
    "precision": 1.0,
    "recall": 1.0,
    "false_positive_rate": 0.0,
    "note": "Sanity check only \u2014 expected 1.0 by construction, confirms no bug"
  },
  "ranking_quality_avg": {
    "precision_at_3": 0.592,
    "recall_at_3": 0.903
  },
  "per_job_ranking_breakdown": [
    {
      "job_id": 101,
      "job_title": "Data Analyst",
      "precision_at_k": 1.0,
      "recall_at_k": 0.375
    },
    {
      "job_id": 102,
      "job_title": "Backend Developer",
      "precision_at_k": 1.0,
      "recall_at_k": 1.0
    },
    {
      "job_id": 103,
      "job_title": "ML Engineer",
      "precision_at_k": 0.333,
      "recall_at_k": 1.0
    },
    {
      "job_id": 104,
      "job_title": "BI Analyst",
   

## Cell 11 — Hand-off Note for Week-3 Monitoring Team

In [11]:
print('HAND-OFF: Baseline for Week-3 monitoring')
print('=' * 55)
print('Current mode: TEST MODE (no real-money payment logic exists yet in this scope)')
print('This snapshot represents ranking quality with zero payment-related friction.')
print()
print('Once gateway is wired and pricing/payment gates affect who applies or who gets')
print('shortlisted, re-run this notebook on the same matches.csv structure and compare:')
print(f"  - Precision@3 (baseline: {per_job_ranking['precision_at_k'].mean():.3f}) — is ranking still surfacing good matches?")
print(f"  - Recall@3 (baseline: {per_job_ranking['recall_at_k'].mean():.3f}) — are fewer good matches reachable in top-3?")
print()
print('A significant drop in either after payments launch = regression, needs review.')
print('(The decision-logic consistency check is NOT used for regression comparison —')
print(' it only confirms the scoring function itself has no bugs.)')

HAND-OFF: Baseline for Week-3 monitoring
Current mode: TEST MODE (no real-money payment logic exists yet in this scope)
This snapshot represents ranking quality with zero payment-related friction.

Once gateway is wired and pricing/payment gates affect who applies or who gets
shortlisted, re-run this notebook on the same matches.csv structure and compare:
  - Precision@3 (baseline: 0.592) — is ranking still surfacing good matches?
  - Recall@3 (baseline: 0.903) — are fewer good matches reachable in top-3?

A significant drop in either after payments launch = regression, needs review.
(The decision-logic consistency check is NOT used for regression comparison —
 it only confirms the scoring function itself has no bugs.)
